*0.3 Classical NLP*

# GloVe

**The situation.** You want word vectors for general English, today, without training anything. Stanford trained GloVe on 6 billion words of Wikipedia and news and published the vectors. Download once, use forever, no GPU, no API.

**GloVe.** Instead of a sliding window, it builds one big table of how often every word appears near every other word across the whole corpus, then fits vectors so that their dot products match the logs of those counts. Different training method from Word2Vec, same kind of result: one vector per word, neighbours by meaning.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import gensim.downloader as api

glove = api.load(
    "glove-wiki-gigaword-50"
)  # 400,000 words × 50 numbers, ~66 MB, cached after the first load
print("words:", len(glove.key_to_index), "| vector size:", glove.vector_size)
for word in ("invoice", "refund", "python"):
    neighbours = []
    for neighbour, score in glove.most_similar(word, topn=4):
        neighbours.append(f"{neighbour} ({score:.2f})")
    print(f"{word:<8} → " + ", ".join(neighbours))
assert glove.similarity("invoice", "receipt") > glove.similarity("invoice", "python")

words: 400000 | vector size: 50
invoice  → 25-cent (0.65), coupon (0.64), receipt (0.63), debit (0.62)
refund   → refunds (0.92), payment (0.81), reimbursement (0.77), payments (0.76)
python   → reticulated (0.69), spamalot (0.66), php (0.64), owl (0.63)


**Reading the output.** General-English neighbours from a vocabulary of 400,000 words — and "python" shows the one-vector-per-word limit: the snake and the language share a single vector, so the neighbours mix.

**The famous arithmetic.** Vector directions carry relations: king − man + woman ≈ queen. It works because GloVe fits ratios of co-occurrence, and ratios encode "the same difference".

In [3]:
result = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
print("king − man + woman ≈", result[0][0], f"({result[0][1]:.2f})")
assert result[0][0] == "queen"

king − man + woman ≈ queen (0.85)


**The rule to remember.** GloVe = pretrained general-English word vectors, free and offline. Good for a quick baseline, features for a classical model, or a vocabulary-similarity lookup.

| Use it when | Don't when | Instead use |
|---|---|---|
| general English words, offline, no training budget | domain jargon (not in the 400k), sentences, context-dependent meaning | Word2Vec on your corpus; sentence embeddings |

**Watch out**
- Trained on 2014 text: no "COVID", no recent product names.
- Bias in the training text is in the vectors (occupation–gender directions are the documented example). Do not use for decisions about people.
- The 50-dimension version is for demos; 300 dimensions is the usual choice.